# Premier League match data without FBref
FBref blocks scrapers and dropped its advanced stats in Jan 2026, so this notebook uses **football-data.co.uk** instead: plain CSV files, no scraping or bot protection. It gives results plus shots, shots on target, fouls, corners and cards per match, reshaped to one row per team per match like the original.

It doesn't have `Dist`, `FK`, `PK` or `PKatt`. The optional last section pulls those (and xG) from **Understat** via `soccerdata`.

In [3]:
import io
from datetime import date

import numpy as np
import pandas as pd
import requests

In [4]:
LEAGUE = "E0"      # E0 = Premier League, E1 = Championship, SP1 = La Liga, D1 = Bundesliga, ...
N_SEASONS = 5

# Season label = year the season ends (2021-22 -> 2022), same as the original notebook
today = date.today()
current = today.year + 1 if today.month >= 8 else today.year
years = list(range(current, current - N_SEASONS, -1))

def season_url(year, league=LEAGUE):
    code = f"{str(year - 1)[-2:]}{str(year)[-2:]}"   # 2026 -> "2526"
    return f"https://www.football-data.co.uk/mmz4281/{code}/{league}.csv"

def fetch_season(year):
    resp = requests.get(season_url(year), timeout=30,
                        headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.content.decode("latin-1")))
    return df.dropna(subset=["HomeTeam", "AwayTeam"])

## Explore one season

In [5]:
raw = fetch_season(years[0])
raw.head()

,ï»¿Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BFEC<2.5,AHCh,B365CAHH,B365CAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,21/08/2026,20:00,Arsenal,Coventry,3,0,H,2,0,...,2.56,-2.00,2.05,1.75,2.05,1.83,2.02,1.74,2.14,1.86
1,E0,22/08/2026,12:30,Hull,Man United,2,0,H,2,0,...,2.56,1.50,1.88,1.98,1.88,1.98,1.82,1.92,1.92,2.06
2,E0,22/08/2026,15:00,Everton,Crystal Palace,2,0,H,1,0,...,1.92,-0.25,2.05,1.80,2.05,1.84,1.96,1.79,2.08,1.90
3,E0,22/08/2026,15:00,Ipswich,Sunderland,2,1,H,1,1,...,1.76,0.00,2.10,1.78,2.10,1.83,1.99,1.77,2.11,1.89
4,E0,22/08/2026,15:00,Nott'm Forest,Leeds,0,1,A,0,0,...,1.77,-0.25,2.10,1.70,2.10,1.75,2.07,1.70,2.19,1.83


## Reshape to one row per team per match

In [6]:
def to_team_rows(df, year):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
    df = df.dropna(subset=["FTHG", "FTAG"])   # skip unplayed fixtures

    def side(me, opp, venue):
        m, o = ("H", "A") if venue == "Home" else ("A", "H")
        out = pd.DataFrame({
            "date": df["Date"], "season": year, "team": df[f"{me}Team"],
            "opponent": df[f"{opp}Team"], "venue": venue,
            "gf": df["FTHG" if venue == "Home" else "FTAG"],
            "ga": df["FTAG" if venue == "Home" else "FTHG"],
            "sh": df[f"{m}S"], "sot": df[f"{m}ST"],
            "opp_sh": df[f"{o}S"], "opp_sot": df[f"{o}ST"],
            "fouls": df[f"{m}F"], "corners": df[f"{m}C"],
            "yellow": df[f"{m}Y"], "red": df[f"{m}R"],
        })
        return out

    rows = pd.concat([side("Home", "Away", "Home"), side("Away", "Home", "Away")])
    rows["result"] = np.select([rows.gf > rows.ga, rows.gf < rows.ga], ["W", "L"], "D")
    return rows.sort_values(["team", "date"]).reset_index(drop=True)

to_team_rows(raw, years[0]).head()

,date,season,team,opponent,venue,gf,ga,sh,sot,opp_sh,opp_sot,fouls,corners,yellow,red,result
0,2026-08-21,2027,Arsenal,Coventry,Home,3,0,20,6,4,1,10,8,1,0,W
1,2026-08-31,2027,Arsenal,Aston Villa,Away,1,0,7,2,7,0,6,4,1,0,W
2,2026-09-06,2027,Arsenal,Chelsea,Home,2,1,16,9,13,5,13,5,2,0,W
3,2026-09-12,2027,Arsenal,Sunderland,Away,2,0,11,5,12,3,11,2,2,0,W
4,2026-09-19,2027,Arsenal,Brighton,Away,0,3,11,2,17,5,11,6,1,0,L


## All seasons

In [7]:
frames = []
for year in years:
    try:
        frames.append(to_team_rows(fetch_season(year), year))
        print("ok", year)
    except (requests.HTTPError, KeyError) as e:
        print("skipped", year, e)

match_df = pd.concat(frames, ignore_index=True)
match_df

ok 2027
ok 2026
ok 2025
ok 2024
ok 2023


,date,season,team,opponent,venue,gf,ga,sh,sot,opp_sh,opp_sot,fouls,corners,yellow,red,result
0,2026-08-21,2027,Arsenal,Coventry,Home,3,0,20,6,4,1,10,8,1,0,W
1,2026-08-31,2027,Arsenal,Aston Villa,Away,1,0,7,2,7,0,6,4,1,0,W
2,2026-09-06,2027,Arsenal,Chelsea,Home,2,1,16,9,13,5,13,5,2,0,W
3,2026-09-12,2027,Arsenal,Sunderland,Away,2,0,11,5,12,3,11,2,2,0,W
4,2026-09-19,2027,Arsenal,Brighton,Away,0,3,11,2,17,5,11,6,1,0,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3135,2023-04-29,2023,Wolves,Brighton,Away,0,6,10,2,22,8,11,5,1,0,L
3136,2023-05-06,2023,Wolves,Aston Villa,Home,1,0,6,2,16,3,20,5,3,0,W
3137,2023-05-13,2023,Wolves,Man United,Away,0,2,5,0,27,9,9,7,2,0,L
3138,2023-05-20,2023,Wolves,Everton,Home,1,1,13,5,19,4,18,1,5,0,D


In [16]:
match_df.tail()

,date,season,team,opponent,venue,gf,ga,sh,sot,opp_sh,opp_sot,fouls,corners,yellow,red,result
3135,2023-04-29,2023,Wolves,Brighton,Away,0,6,10,2,22,8,11,5,1,0,L
3136,2023-05-06,2023,Wolves,Aston Villa,Home,1,0,6,2,16,3,20,5,3,0,W
3137,2023-05-13,2023,Wolves,Man United,Away,0,2,5,0,27,9,9,7,2,0,L
3138,2023-05-20,2023,Wolves,Everton,Home,1,1,13,5,19,4,18,1,5,0,D
3139,2023-05-28,2023,Wolves,Arsenal,Away,0,5,6,0,14,8,11,4,0,0,L


In [8]:
# match_df.to_csv("matches.csv", index=False)

## Optional: Dist / FK / PK / xG from Understat
Understat has shot-level data (location, situation, xG) for the top European leagues. **Untested:** I couldn't run this here, so check the column names against `shots.columns` and the `soccerdata` docs. Team names differ between Understat and football-data.co.uk, so you'll need a name mapping before merging.

In [1]:
pip install soccerdata

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:

import soccerdata as sd

us = sd.Understat("ENG-Premier League", seasons=[f"{str(y-1)[-2:]}{str(y)[-2:]}" for y in years])
shots = us.read_shot_events().reset_index()
print(shots.columns.tolist())
shots.head()

[09/29/26 17:22:52] INFO     Saving cached data to C:\Users\lenovo\soccerdata\data\Understat         ]8;id=12342944;file://C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12342945;file://C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-29 17:22:52] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=12342952;file://C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12342953;file://C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\tls_re                 
                             quests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                                    

['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']


,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,...,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
0,ENG-Premier League,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,Bukayo Saka,1,2022,18202,2022-08-05 19:00:00,479033,...,7322,539138,Ben White,0.078887,0.96,0.328,69,Right Foot,Open Play,Missed Shot
1,ENG-Premier League,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,Bukayo Saka,1,2022,18202,2022-08-05 19:00:00,479038,...,7322,539142,Thomas Partey,0.057627,0.95,0.277,84,Right Foot,Open Play,Missed Shot
2,ENG-Premier League,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,Bukayo Saka,1,2022,18202,2022-08-05 19:00:00,479040,...,7322,539149,Eddie Nketiah,0.096064,0.929,0.33,86,Right Foot,Open Play,Blocked Shot
3,ENG-Premier League,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,Gabriel Jesus,1,2022,18202,2022-08-05 19:00:00,479020,...,5543,<NA>,<NA>,0.181834,0.883,0.507,3,Right Foot,Open Play,Blocked Shot
4,ENG-Premier League,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,Gabriel Martinelli,1,2022,18202,2022-08-05 19:00:00,479021,...,7752,<NA>,<NA>,0.340446,0.935,0.612,3,Left Foot,Open Play,Missed Shot


In [10]:
# Distance to goal centre in metres, assuming location_x/y are 0-1 and the pitch is 105 x 68 m
shots["dist"] = np.sqrt(((1 - shots["location_x"]) * 105) ** 2 + ((0.5 - shots["location_y"]) * 68) ** 2)

per_game = (shots.groupby(["season", "game", "team"])
    .agg(shots=("xg", "size"), xg=("xg", "sum"), avg_dist=("dist", "mean"),
         fk=("situation", lambda s: (s == "DirectFreekick").sum()),
         pk=("situation", lambda s: (s == "Penalty").sum()))
    .reset_index())
per_game.head()

,season,game,team,shots,xg,avg_dist,fk,pk
0,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,10,1.497916,15.573992,0,0
1,2223,2022-08-05 Crystal Palace-Arsenal,Crystal Palace,11,1.210105,22.546641,0,0
2,2223,2022-08-06 Bournemouth-Aston Villa,Aston Villa,15,0.488895,22.876048,0,0
3,2223,2022-08-06 Bournemouth-Aston Villa,Bournemouth,7,0.592635,16.597716,0,0
4,2223,2022-08-06 Everton-Chelsea,Chelsea,15,1.923145,14.92142,0,0


In [18]:
per_game.head()

,season,game,team,shots,xg,avg_dist,fk,pk
0,2223,2022-08-05 Crystal Palace-Arsenal,Arsenal,10,1.497916,15.573992,0,0
1,2223,2022-08-05 Crystal Palace-Arsenal,Crystal Palace,11,1.210105,22.546641,0,0
2,2223,2022-08-06 Bournemouth-Aston Villa,Aston Villa,15,0.488895,22.876048,0,0
3,2223,2022-08-06 Bournemouth-Aston Villa,Bournemouth,7,0.592635,16.597716,0,0
4,2223,2022-08-06 Everton-Chelsea,Chelsea,15,1.923145,14.92142,0,0


In [ ]:
match_df.head(
    
)

In [ ]:
shots.head()